In [ ]:
# Install dependencies
!pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm joblib

# Lifeboat: Titanic Survival Prediction

Complete notebook: EDA, model training, and inference.

---

In [ ]:
# Imports
import warnings

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

TITANIC_URL = (
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
)
MODEL_DIR = "models"

## 1. Load Data

In [ ]:
# Load data
df = pd.read_csv(TITANIC_URL)
print(f"Dataset shape: {df.shape}")
df.head()

## 2. EDA

In [ ]:
# Basic dataset info
print("=" * 50)
print("DATASET INFO")
print("=" * 50)
print(f"Total passengers: {len(df)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nSurvival rate: {df['Survived'].mean()*100:.1f}%")

In [ ]:
# Survival distribution
fig, ax = plt.subplots(figsize=(6, 5))
df["Survived"].value_counts().plot(
    kind="pie",
    autopct="%1.1f%%",
    colors=["#ff6b6b", "#51cf66"],
    labels=["Did not survive", "Survived"],
    explode=(0, 0.05),
    ax=ax,
)
ax.set_title("Survival Distribution", fontsize=14, fontweight="bold")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Survival by Sex and Class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sex_survival = df.groupby("Sex")["Survived"].mean() * 100
sex_survival.plot(kind="bar", color=["#339af0", "#ff6b6b"], ax=axes[0])
axes[0].set_title("Survival Rate by Sex", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Survival Rate (%)")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

pclass_survival = df.groupby("Pclass")["Survived"].mean() * 100
pclass_survival.plot(kind="bar", ax=axes[1], color=["#ffd43b", "#ffe066", "#fcc419"])
axes[1].set_title("Survival Rate by Passenger Class", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Survival Rate (%)")
axes[1].set_xticklabels(["1st", "2nd", "3rd"], rotation=0)

plt.tight_layout()
plt.show()

print(f"Women survival: {df[df['Sex']=='female']['Survived'].mean()*100:.1f}%")
print(f"Men survival: {df[df['Sex']=='male']['Survived'].mean()*100:.1f}%")

In [ ]:
# Age and Fare distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["Age"].hist(bins=30, ax=axes[0], color="#339af0", edgecolor="white")
axes[0].set_title("Age Distribution", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")

df["Fare"].hist(bins=30, ax=axes[1], color="#fcc419", edgecolor="white")
axes[1].set_title("Fare Distribution", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Fare")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
numeric_cols = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="RdYlGn", center=0, fmt=".2f", linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nTop correlations with Survival:")
print(corr["Survived"].sort_values(ascending=False))

## 3. Preprocessing

In [ ]:
# Feature engineering
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
df["FarePerPerson"] = df["Fare"] / df["FamilySize"]
df["AgeBin"] = pd.cut(
    df["Age"], bins=[0, 12, 18, 35, 60, 100], labels=[0, 1, 2, 3, 4]
).astype(float)

# Encode categorical variables
le_sex = LabelEncoder()
le_embarked = LabelEncoder()
df["Sex_encoded"] = le_sex.fit_transform(df["Sex"])
df["Embarked_encoded"] = le_embarked.fit_transform(df["Embarked"])

print("Feature engineering complete!")
print(
    f"Label mappings: Sex={dict(zip(le_sex.classes_, le_sex.transform(le_sex.classes_)))}"
)
print(
    f"Label mappings: Embarked={\
dict(zip(le_embarked.classes_, le_embarked.transform(le_embarked.classes_)))}"
)

In [ ]:
# Define features and target
features = [
    "Pclass",
    "Sex_encoded",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked_encoded",
    "FamilySize",
    "IsAlone",
    "FarePerPerson",
    "AgeBin",
]

X = df[features]
y = df["Survived"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"Features: {features}")

## 4. Training

In [ ]:
# Train GradientBoosting model
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42
)

# Cross-validation
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="accuracy")
print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Train
model.fit(X_train_scaled, y_train)

# Test predictions
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

# Metrics
test_acc = accuracy_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_proba)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test AUC-ROC: {test_auc:.4f}")

In [ ]:
# Classification report
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=["Did not survive", "Survived"]))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=ax,
    xticklabels=["Did not survive", "Survived"],
    yticklabels=["Did not survive", "Survived"],
)
ax.set_title("Confusion Matrix - GradientBoosting", fontsize=14, fontweight="bold")
ax.set_ylabel("Actual")
ax.set_xlabel("Predicted")
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importances = model.feature_importances_
feat_imp = pd.DataFrame({"Feature": features, "Importance": importances})
feat_imp = feat_imp.sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_imp["Feature"], feat_imp["Importance"], color="#51cf66")
ax.set_xlabel("Importance")
ax.set_title("Feature Importance - GradientBoosting", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nTop 5 Important Features:")
print(feat_imp.tail(5).to_string(index=False))

## 5. Save Model

In [ ]:
# Save model and artifacts
import os

os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(model, f"{MODEL_DIR}/titanic_model.pkl")
joblib.dump(scaler, f"{MODEL_DIR}/scaler.pkl")
joblib.dump(features, f"{MODEL_DIR}/features.pkl")
joblib.dump(le_sex, f"{MODEL_DIR}/le_sex.pkl")
joblib.dump(le_embarked, f"{MODEL_DIR}/le_embarked.pkl")

print(f"Model saved to: {MODEL_DIR}/titanic_model.pkl")
print(f"Scaler saved to: {MODEL_DIR}/scaler.pkl")

## 6. Inference

In [ ]:
# Load artifacts
model = joblib.load(f"{MODEL_DIR}/titanic_model.pkl")
scaler = joblib.load(f"{MODEL_DIR}/scaler.pkl")
features = joblib.load(f"{MODEL_DIR}/features.pkl")
le_sex = joblib.load(f"{MODEL_DIR}/le_sex.pkl")
le_embarked = joblib.load(f"{MODEL_DIR}/le_embarked.pkl")

print("Model artifacts loaded successfully!")

In [ ]:
# Prediction function
def predict_survival(pclass, sex, age, sibsp, parch, fare, embarked):
    """Predict Titanic survival probability."""
    family_size = sibsp + parch + 1
    is_alone = 1 if family_size == 1 else 0
    fare_per_person = fare / family_size

    age_bin = 0
    if age > 12:
        age_bin = 1
    if age > 18:
        age_bin = 2
    if age > 35:
        age_bin = 3
    if age > 60:
        age_bin = 4

    sample = pd.DataFrame(
        {
            "Pclass": [pclass],
            "Sex_encoded": [le_sex.transform([sex])[0]],
            "Age": [age],
            "SibSp": [sibsp],
            "Parch": [parch],
            "Fare": [fare],
            "Embarked_encoded": [le_embarked.transform([embarked])[0]],
            "FamilySize": [family_size],
            "IsAlone": [is_alone],
            "FarePerPerson": [fare_per_person],
            "AgeBin": [age_bin],
        }
    )

    sample_scaled = scaler.transform(sample)
    survival = int(model.predict(sample_scaled)[0])
    probability = float(model.predict_proba(sample_scaled)[0][1])

    return {"survival": survival, "probability": probability}

In [ ]:
# Example predictions
print("=" * 50)
print("EXAMPLE PREDICTIONS")
print("=" * 50)

# Example 1: 3rd class male, 25 years old
result1 = predict_survival(
    pclass=3, sex="male", age=25, sibsp=0, parch=0, fare=7.25, embarked="S"
)
print(f"\nPassenger 1 (3rd class, male, 25): {result1}")

# Example 2: 1st class female, 30 years old
result2 = predict_survival(
    pclass=1, sex="female", age=30, sibsp=1, parch=1, fare=100, embarked="C"
)
print(f"Passenger 2 (1st class, female, 30): {result2}")

# Example 3: 2nd class male, 40 years old
result3 = predict_survival(
    pclass=2, sex="male", age=40, sibsp=1, parch=0, fare=30, embarked="Q"
)
print(f"Passenger 3 (2nd class, male, 40): {result3}")

---*Notebook complete! Model is ready for deployment.*